# Interpolation with a shared family frame and cap phase

Does the large interpolation error in notebook 04 persist when known boundary variants share a physical origin, orientation, and consistent angular phase?

We repeat the **same two crypts, boundary cuts, degree-12 basis, size conventions, and coefficient mixtures**. The old method independently aligns each cut to the frozen cohort template. The new method aligns the baseline once, then uses that baseline's physical frame for its entire family. Shared original membrane vertices in the unchanged core determine a single cap-phase rotation. No detector tip, biological axis, chart warping, or new surface fit enters that correction.

Physical-size variants retain source lengths. Unit-RMS variants retain **their own measured radius**, exactly as before: normalization can still change with the opening. Both methods use the same baseline reference and evaluation points. Generated meshes are never realigned afterward.

Run notebook 04 first, and use the `organoid-mesh` environment. This notebook reads its settings to freeze the cases and regenerates the fits/checks. Results are written separately to `exports/crypt_morphology/family_alignment/`. Start with the paired summary and error curves; the detailed CSVs are for follow-up inspection.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/organograph").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import display
pio.renderers.default = "plotly_mimetype"
from organograph.crypt_morphology import *
from organograph.crypt_morphology.datasets import discover_hks_crypts, load_hks_parent
from organograph.plotting.crypt_morphology import plot_decoding_comparison, plot_alignment_interpolation_response
ARTIFACTS = ROOT / "exports/crypt_morphology"
PREVIOUS = ARTIFACTS / "decoding_stability"
OUT = ARTIFACTS / "family_alignment"
OUT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get("ORGANOGRAPH_DATA_ROOT", ROOT.parent / "NicoleData"))
previous = json.loads((PREVIOUS / "settings.json").read_text())
SIZE_MODES = tuple(previous["size_modes"])
GRID, FINE_GRID = tuple(previous["standard_grid"]), tuple(previous["fine_grid"])
FRACTIONS, PATHS = previous["interpolation_fractions"], previous["paths"]
METHODS = {"independent": "Independent alignment", "shared": "Shared frame + phase"}
PLOT_SIZE_MODE = "physical"  # Change to unit_rms to inspect the size-normalized meshes.
PLOT_PATHS = ("uniform",)  # Both paths are tested; add "sideways" to display its meshes.
basis = load_basis(ARTIFACTS / "pilot_basis.json")
assert basis.identifier == previous["basis_id"]
references = {mode: load_alignment_reference(ARTIFACTS / f"pilot_alignment_{mode}.npz", basis) for mode in SIZE_MODES}
for mode in SIZE_MODES:
    digest = hashlib.sha256((ARTIFACTS / f"pilot_alignment_{mode}.npz").read_bytes()).hexdigest()
    assert digest == previous["alignment_sha256"][mode], "Cohort reference changed since notebook 04"
def key_of(record):
    return (str(record["dataset"]), record["label_uid"], int(record["crypt_index"]))
records = {key_of(r): r for r in discover_hks_crypts(DATA_ROOT)}

## Hold the physical frame fixed and use actual correspondence for cap phase

The unchanged core contains baseline triangles more than `0.1 R₀` inward from the original rim. Its original vertices have the same source IDs and physical coordinates in every cut. New vertices inserted along a cut are excluded. Fixed physical-area weights come from those same baseline core triangles.

For complex disk coordinates `z = u + i v`, fit one angle δ minimizing `Σ w |z_variant exp(−iδ) − z_baseline|²`. This removes an arbitrary angular start without searching for a new 3D pose. Points near the pole contribute less because their disk radius is small. The residual records radial/nonrigid chart changes which one rotation cannot remove. A coherence near one means the shared points agree about the angle; it does **not** mean the charts are identical.

The resulting coefficients undergo an exact pose/phase transformation. Assertions below verify that restoring source coordinates recovers the same fitted surface, that both methods keep the same scale, and that every shared-family variant has the baseline's physical origin and rotation. The same source geometry must be available for this control; this is not an alignment method for unrelated crypts.

In [ ]:
cases, fit_rows, phase_rows = {}, [], []
q_check, _ = basis.reference_mesh(6, 24)
for case, saved in previous["cases"].items():
    record = records[key_of(saved["source"])]
    v, f, ids, neighbors, metadata = load_hks_parent(record)
    assert metadata["mesh_geometry_sha256"] == saved["source"]["mesh_geometry_sha256"]
    patch = extract_patch(v, f, ids)
    mapping = parameterize_patch(patch, half_angle=basis.half_angle)
    baseline_raw = fit_representation(patch, mapping, basis, metadata=metadata)
    radius = baseline_raw.rms_radius
    family = prepare_boundary_family(v, f, ids, max_displacement=.1*radius,
                                     collar_width=.3*radius, forbidden_vertex_ids=neighbors)
    patches, mappings, raw = {"baseline": patch}, {"baseline": mapping}, {"baseline": baseline_raw}
    fit_rows.append(dict(case=case, variant="baseline", status="ok", error=""))
    for shift in opening_shifts(radius):
        if shift.name == "baseline":
            continue
        try:
            cut = family.cut(shift).patch
            mapped = parameterize_patch(cut, half_angle=basis.half_angle)
            rep = fit_representation(cut, mapped, basis)
        except ValueError as error:
            fit_rows.append(dict(case=case, variant=shift.name, status="rejected_or_failed", error=str(error)))
            continue
        patches[shift.name], mappings[shift.name], raw[shift.name] = cut, mapped, rep
        fit_rows.append(dict(case=case, variant=shift.name, status="ok", error=""))
    core_mask = family.signed_distance[f[patch.source_face_ids]].max(axis=1) < -.1*radius
    weights = vertex_areas(patch.vertices, patch.faces[core_mask])
    active = weights > 0
    core_ids, core_weights = patch.source_vertex_ids[active], weights[active]
    np.savez_compressed(OUT / f"{case.split()[0].lower()}_core.npz", source_vertex_ids=core_ids,
                        weights=core_weights, source_face_ids=patch.source_face_ids[core_mask])
    by_mode = {}
    for mode in SIZE_MODES:
        names = list(raw)
        scaled = {name: with_size_mode(raw[name], mode) for name in names}
        aligned = align_to_reference(list(scaled.values()), basis, references[mode])
        assert aligned.converged
        independent = dict(zip(names, aligned.representations))
        baseline = independent["baseline"]
        shared = {}
        for name in names:
            rep, phase = align_known_boundary_variant(scaled[name], patches[name], mappings[name],
                baseline, patch, mapping, basis, core_ids, core_weights)
            shared[name] = rep
            np.testing.assert_array_equal(rep.pose.rotation, baseline.pose.rotation)
            origin_difference = np.linalg.norm(rep.pose.inverse(np.zeros(3))-baseline.pose.inverse(np.zeros(3)))
            assert origin_difference < 1e-10*radius
            assert rep.pose.scale == independent[name].pose.scale == scaled[name].pose.scale
            old_world = scaled[name].pose.inverse(basis.evaluate(q_check-[0, scaled[name].phase]) @ scaled[name].coefficients)
            new_world = rep.pose.inverse(basis.evaluate(q_check-[0, rep.phase]) @ rep.coefficients)
            source_difference = np.max(np.linalg.norm(old_world-new_world, axis=1))/radius
            assert source_difference < 1e-10, "Reframing changed the fitted surface"
            rotation = independent[name].pose.rotation @ baseline.pose.rotation.T
            phase_difference = np.angle(np.exp(1j*(independent[name].phase-rep.phase)))
            phase_rows.append(dict(case=case, size_mode=mode, variant=name,
                phase_offset=phase.phase_offset, phase_coherence=phase.coherence,
                disk_rmse_before=phase.disk_rmse_before, disk_rmse_after=phase.disk_rmse_after,
                n_core_vertices=phase.n_vertices, independent_phase_difference_degrees=np.degrees(phase_difference),
                independent_pose_rotation_degrees=np.degrees(np.arccos(np.clip((np.trace(rotation)-1)/2, -1, 1))),
                scale_ratio=rep.pose.scale/baseline.pose.scale, shared_origin_error_relative=origin_difference/radius,
                source_geometry_change_relative=source_difference))
        np.testing.assert_allclose(shared["baseline"].coefficients, baseline.coefficients, atol=1e-10)
        by_mode[mode] = dict(independent=independent, shared=shared)
    cases[case] = dict(patch=patch, mapping=mapping, raw=raw, aligned=by_mode,
                       core_mask=core_mask, radius=radius, metadata=metadata, boundary=family.diagnostics)
    print(f"{case}: {len(raw)}/17 fitted openings, {len(core_ids)} fixed core vertices")
fit_table, phases = pd.DataFrame(fit_rows), pd.DataFrame(phase_rows)
fit_table.to_csv(OUT / "opening_fits.csv", index=False)
phases.to_csv(OUT / "phase_diagnostics.csv", index=False)
if (fit_table.status != "ok").any():
    display(fit_table[fit_table.status != "ok"])
# Compare only; previous numerical outputs are never used as current results.
with np.load(PREVIOUS / "decoded_test_coefficients.npz", allow_pickle=False) as old:
    old_coefficients = {(str(c), str(m), str(s)): a for c, m, s, a in
                        zip(old["case"], old["size_mode"], old["scenario"], old["coefficients"])}
max_repeat_difference = 0.
for case, data in cases.items():
    for mode in SIZE_MODES:
        for name, rep in data["aligned"][mode]["independent"].items():
            difference = np.max(np.abs(rep.coefficients-old_coefficients[case, mode, name]))
            max_repeat_difference = max(max_repeat_difference, difference)
            # Numerical phase minimization can shift the last digits between runs.
            # Require agreement far below the reconstruction errors (1e-6 of R0).
            radius_in_units = data["radius"]/rep.pose.scale
            np.testing.assert_allclose(rep.coefficients, old_coefficients[case, mode, name],
                                       atol=1e-6*radius_in_units, rtol=0)
            grid_delta = basis.evaluate(q_check) @ (rep.coefficients-old_coefficients[case, mode, name])
            assert np.linalg.norm(grid_delta, axis=1).max()/radius_in_units < 1e-6
print(f"Prior independent-alignment coefficients reproduced (largest absolute difference: {max_repeat_difference:.3g}).")
display(phases[phases.size_mode == "physical"].groupby("case").agg(
    minimum_phase_coherence=("phase_coherence", "min"), maximum_disk_residual=("disk_rmse_after", "max"),
    largest_previous_rotation_degrees=("independent_pose_rotation_degrees", "max")).round(4))

## Repeat the interpolation and geometry checks

Each path joins opposite `0.1 R₀` cuts: uniform inward/outward movement, or opposite sideways shifts. We decode mixtures at 25%, 50%, and 75%. All 17 observed cuts are checked too. The primary error is **distance from actual, unchanged source-core triangle centroids to the generated surface**, divided by baseline RMS radius. Centroids are distinct from the fitting quadrature points. These are geometric errors, not latent-code distances. RMS averages can hide local deviations, so we also print the largest sampled core distance; it is not a certified maximum over the continuous surface.

The same core and baseline pose are used for both methods. A fixed cap-sample displacement is also saved, but can include chart movement and is not the main accuracy measure. In unit-RMS mode, differences in per-cut radius still contribute to error; the test deliberately keeps that convention unchanged.

All outputs are checked for triangle intersections, collapsed faces, and normal turns exceeding 90° relative to the baseline. Baselines, all four endpoints, both midpoints, the worst interpolation on each path, and all flagged outputs are repeated on the finer grid. Normal turns are warnings, not proof of a fold. These finite-grid checks cannot certify all continuous surfaces or arbitrary coefficients.

In [ ]:
probes, coefficients, scenario_info, rows, skipped = {}, {}, {}, [], []
for case, data in cases.items():
    for mode in SIZE_MODES:
        baseline = data["aligned"][mode]["independent"]["baseline"]
        probe = DecodingProbe(data["patch"], data["mapping"], baseline, basis, data["core_mask"],
                              n_rings=GRID[0], n_sectors=GRID[1])
        probes[case, mode] = probe
        for method in METHODS:
            fitted = data["aligned"][mode][method]
            scenarios = [(name, rep.coefficients, dict(kind="observed_variant", variant=name)) for name, rep in fitted.items()]
            for path, (left, right) in PATHS.items():
                if left not in fitted or right not in fitted:
                    skipped.append(dict(case=case, size_mode=mode, method=method, path=path,
                                        reason="Required endpoint rejected; no interpolation performed"))
                    continue
                for fraction in FRACTIONS:
                    name = f"blend_{path}_{fraction:.2f}"
                    c = interpolate_coefficients(fitted[left].coefficients, fitted[right].coefficients, fraction, basis)
                    scenarios.append((name, c, dict(kind="interpolation", path=path, fraction=fraction,
                                                    endpoint_a=left, endpoint_b=right)))
            for name, c, info in scenarios:
                key = (case, mode, method, name)
                coefficients[key], scenario_info[key] = c, info
                measurements, _ = probe.evaluate(c)
                rows.append(dict(case=case, size_mode=mode, method=method, scenario=name, **info, **measurements))
        print(f"Standard checks completed: {case}, {mode}")
results = pd.DataFrame(rows)
results.to_csv(OUT / "decoding_checks.csv", index=False)
if skipped:
    display(pd.DataFrame(skipped))
keys = list(coefficients)
np.savez_compressed(OUT / "decoded_test_coefficients.npz", coefficients=np.stack([coefficients[k] for k in keys]),
                    case=np.array([k[0] for k in keys]), size_mode=np.array([k[1] for k in keys]),
                    method=np.array([k[2] for k in keys]), scenario=np.array([k[3] for k in keys]), basis_id=basis.identifier)
fine_rows = []
for case, data in cases.items():
    for mode in SIZE_MODES:
        probe = DecodingProbe(data["patch"], data["mapping"], data["aligned"][mode]["independent"]["baseline"],
                              basis, data["core_mask"], n_rings=FINE_GRID[0], n_sectors=FINE_GRID[1])
        for method in METHODS:
            group = results[(results.case == case) & (results.size_mode == mode) & (results.method == method)]
            names = {"baseline", *[name for pair in PATHS.values() for name in pair]}
            for path in PATHS:
                names.add(f"blend_{path}_0.50")
                part = group[(group.kind == "interpolation") & (group.path == path)]
                if len(part):
                    names.add(part.loc[part.core_surface_rmse_relative.idxmax(), "scenario"])
            names.update(group[group.geometry_warning].scenario)
            for name in sorted(names):
                key = (case, mode, method, name)
                if key not in coefficients:
                    continue
                measurements, _ = probe.evaluate(coefficients[key])
                fine_rows.append(dict(case=case, size_mode=mode, method=method, scenario=name,
                                      **scenario_info[key], **measurements))
        print(f"Fine checks completed: {case}, {mode}")
fine = pd.DataFrame(fine_rows)
fine.to_csv(OUT / "fine_mesh_checks.csv", index=False)
print(f"Checked {len(results)} standard and {len(fine)} fine meshes across both methods.")

## The paired result

Read each row from baseline fitting error to the worst of six interpolated meshes. Lower error after fixing the frame/phase identifies a contribution from independent alignment choices. Residual error can come from finite basis approximation, remaining nonrigid chart changes, interpolation itself, and size normalization. A ratio or small error alone is not an anatomical acceptance criterion.

The curve endpoints are observed cuts; the middle points are generated coefficient mixtures. Both paths and both size modes are included. Detailed per-path and fine-grid results are saved alongside the small summary.

In [ ]:
summary_rows, path_rows = [], []
for (case, mode), group in results.groupby(["case", "size_mode"], sort=False):
    row = {"Crypt": case, "Size mode": mode,
           "Baseline core error (%)": 100*group[group.scenario == "baseline"].core_surface_rmse_relative.iloc[0]}
    for method, label in METHODS.items():
        part = group[(group.method == method) & (group.kind == "interpolation")]
        row[label+": worst interpolation (%)"] = 100*part.core_surface_rmse_relative.max()
        for path in PATHS:
            selected = part[part.path == path]
            path_rows.append(dict(case=case, size_mode=mode, method=method, path=path,
                                  worst_core_error_percent=100*selected.core_surface_rmse_relative.max()))
    summary_rows.append(row)
summary, path_summary = pd.DataFrame(summary_rows), pd.DataFrame(path_rows)
summary.to_csv(OUT / "summary.csv", index=False)
path_summary.to_csv(OUT / "per_path_summary.csv", index=False)
with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
    display(summary.round(3))
print("RMS error is an average. Largest single sampled core distances among shared-frame mixtures:")
for case, part in results[(results.method == "shared") & (results.kind == "interpolation")].groupby("case"):
    print(f"  {case}: {100*part.core_surface_sampled_max_relative.max():.2f}% of R0 (across both size modes)")
all_checks = pd.concat([results.assign(grid="standard"), fine.assign(grid="fine")], ignore_index=True)
flags = all_checks[all_checks.geometry_warning]
flags.to_csv(OUT / "flagged_meshes.csv", index=False)
print(f"Geometry warnings: {len(flags)} / {len(all_checks)} checked meshes.")
if len(flags):
    display(flags[["case", "size_mode", "method", "scenario", "grid", "intersection_pairs", "collapsed_faces", "normal_turn_faces"]])
paired_grid = fine.merge(results, on=["case", "size_mode", "method", "scenario"], suffixes=("_fine", "_standard"))
print("Largest core-error change on mesh refinement (percentage points):",
      round(100*(paired_grid.core_surface_rmse_relative_fine-paired_grid.core_surface_rmse_relative_standard).abs().max(), 4))
figure = plot_alignment_interpolation_response(results, PATHS, methods=METHODS)
figure.savefig(OUT / "interpolation_errors.png", dpi=160)
display(figure)
plt.close(figure)

## Compare the meshes directly

Each figure has the original independent alignment in the top row and the shared-family frame/phase in the bottom row. Columns run from endpoint A through 25%, 50%, 75% mixtures to endpoint B. Blue is the generated surface, gray is the unchanged baseline, and black marks the opening. All panels in a figure share spatial limits. No centering, scaling, rotation, or smoothing is performed for display.

The figures default to the uniform path in physical units. Change `PLOT_SIZE_MODE` or `PLOT_PATHS` in the first cell to inspect other tested conditions. Standalone interactive HTML and static PNG comparisons for both paths are saved in the chosen size mode.

In [ ]:
for case in cases:
    probe = probes[case, PLOT_SIZE_MODE]
    for path, (left, right) in PATHS.items():
        names = [left, *[f"blend_{path}_{fraction:.2f}" for fraction in FRACTIONS], right]
        if not all((case, PLOT_SIZE_MODE, method, name) in coefficients for method in METHODS for name in names):
            continue
        sequences = {label: [probe.matrix @ coefficients[case, PLOT_SIZE_MODE, method, name] for name in names]
                     for method, label in METHODS.items()}
        labels = ["Endpoint A", *[f"{100*fraction:.0f}% toward B" for fraction in FRACTIONS], "Endpoint B"]
        title = f"{case}: {path} cuts — {PLOT_SIZE_MODE}"
        stem = f"{case.split()[0].lower()}_{path}_{PLOT_SIZE_MODE}"
        fig = plot_decoding_comparison(sequences, probe.faces, labels, reference=probe.baseline_vertices, title=title)
        fig.write_html(OUT / f"{stem}.html", include_plotlyjs=True)
        if path in PLOT_PATHS:
            display(fig)
        static = plot_decoding_comparison(sequences, probe.faces, labels, reference=probe.baseline_vertices,
                                          title=title, static=True)
        static.savefig(OUT / f"{stem}.png", dpi=150)
        plt.close(static)

## Interpretation and limits

This is a controlled test of **known variants from the same membrane**. It asks whether coefficient mixtures become more faithful when arbitrary pose/phase changes are removed before interpolation. It does not establish boundary-invariant latent codes or solve correspondence between different crypts.

A successful result would support keeping a consistent frame/phase within each augmentation family, with the baseline aligned once to the cohort. It would also support testing boundary consistency in the later encoder while reconstructing ordinary, unblurred cap coefficients. Remaining chart differences should stay visible instead of being hidden by a new warp. Per-variant normalization remains a separate sensitivity choice.

Only two crypts and two mixture paths are tested, with no learned decoder and no claim about arbitrary coefficient samples. All variants from an organoid must remain in the same train/test split. Broader validation should keep family consistency separate from the alignment problem across unrelated crypts.

In [ ]:
settings = dict(basis_id=basis.identifier, max_degree=basis.max_degree, standard_grid=GRID, fine_grid=FINE_GRID,
    interpolation_fractions=FRACTIONS, paths=PATHS, size_modes=SIZE_MODES, methods=METHODS,
    alignment="Baseline aligned to frozen cohort; family shares physical origin/rotation; phase from original core IDs",
    phase_objective="sum w abs(z_variant exp(-i delta) - z_baseline)^2",
    phase_weights="fixed baseline core triangle areas lumped to original vertices",
    preserve_variant_scale=True, generated_output_realignment=False,
    core_definition="baseline triangles with all vertices signed edge distance < -0.1*R0",
    cases={case: dict(source=data["metadata"], boundary=data["boundary"]) for case, data in cases.items()},
    previous_settings_sha256=hashlib.sha256((PREVIOUS / "settings.json").read_bytes()).hexdigest(),
    alignment_sha256=previous["alignment_sha256"], max_prior_coefficient_difference=max_repeat_difference,
    incident_contraction=1e-7, triangle_pair_interval_tolerance=1e-12,
    standard_mesh_checks=len(results), fine_mesh_checks=len(fine), skipped=skipped)
(OUT / "settings.json").write_text(json.dumps(settings, indent=2))
save_basis(OUT / "basis.json", basis)
print("Saved reports, coefficients, correspondence weights, settings and figures to", OUT)